# Module C, Week 3: Feature Selection, PCR, and PLSR
### Integrated Capstone Project: Predicting Corporate Bankruptcy from Financial Ratios
**Astha Goel**

---

## Project context

This capstone predicts corporate bankruptcy from published financial ratios, running the same pipeline
separately on two datasets and comparing results rather than merging them:

| | Poland | Taiwan |
|---|---|---|
| Observations | 43,405 firm-years (2000 to 2013) | 6,819 firms (1999 to 2009) |
| Features | 64 financial ratios | 95 financial ratios |
| Bankrupt | 2,091 (4.8%) | 220 (3.2%) |

**Established so far.** Module B: no single ratio predicts bankruptcy alone, PCA does not compress these data
(PC1 captures only 15.3% and 13.4% of variance), yet a random forest reaches 0.9395 (Poland) and 0.9482 (Taiwan).
Week 1: severe multicollinearity (a Taiwanese pair at VIF ≈ 10¹⁵), and added flexibility helps Poland and does not help Taiwan. Week 2: features ranging to 10¹⁰ break linear models until winsorized; lasso removes redundant
ratio families automatically. Week 4: scaler choice swings Poland's ROC-AUC by 0.21, `QuantileTransformer` wins
both datasets, and `class_weight='balanced'` degrades calibration without improving ranking.

## This week

Week 3 covers three ways of reducing a wide feature set: stepwise selection (forward and backward), principal
component regression, and partial least squares regression. All three are directly motivated by what earlier
weeks found, these datasets have 64 and 94 features with severe redundancy, so the obvious question is how few
are actually needed.

The Module B PCA result makes a specific prediction worth testing: since PC1 captures only 15.3% and 13.4% of
variance, compression should *cost* accuracy rather than save it. This notebook tests that directly.

## 1. Setup and loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

# rank-based scaling, selected in Week 4 after comparing six scalers
qt = lambda: QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression

In [ ]:
# ── TAIWAN ────────────────────────────────────────────────────────────────────
df_taiwan = pd.read_csv('Taiwan_data_capstone.csv')
df_taiwan.columns = df_taiwan.columns.str.strip()   # headers carry a leading space
df_taiwan['Bankrupt?'] = pd.to_numeric(df_taiwan['Bankrupt?'], errors='coerce').astype(int)

numeric_cols_taiwan = [c for c in df_taiwan.select_dtypes(include='number').columns
                       if c != 'Bankrupt?']
for col in numeric_cols_taiwan:
    df_taiwan[col] = df_taiwan[col].fillna(df_taiwan[col].median())
numeric_cols_taiwan = [c for c in numeric_cols_taiwan if df_taiwan[c].nunique() > 1]

print(f"Taiwan: {df_taiwan.shape}, {len(numeric_cols_taiwan)} usable features")
print(df_taiwan['Bankrupt?'].value_counts().to_dict())

In [ ]:
# ── POLAND ────────────────────────────────────────────────────────────────────
df_polish = pd.read_csv('Polish_dataset_capstone.csv')

column_names_64 = [
    'net_profit_to_total_assets', 'total_liabilities_to_total_assets',
    'working_capital_to_total_assets', 'current_assets_to_short_term_liabilities',
    'cash_to_operating_expenses', 'retained_earnings_to_total_assets',
    'EBIT_to_total_assets', 'book_value_of_equity_to_total_liabilities',
    'sales_to_total_assets', 'equity_to_total_assets',
    'gross_profit_to_short_term_liabilities', 'gross_profit_to_total_assets',
    'gross_profit_to_sales', 'gross_profit_to_total_assets_2',
    'working_capital_to_fixed_assets', 'receivables_to_total_assets',
    'liabilities_to_total_assets_duration', 'EBIT_to_total_assets_2',
    'sales_to_total_assets_2', 'working_capital_to_sales',
    'sales_to_inventories', 'net_profit_to_sales',
    'gross_profit_to_total_assets_3', 'current_liabilities_to_equity',
    'short_term_liabilities_to_equity', 'working_capital',
    'depreciation_to_total_assets', 'inventories_to_sales',
    'net_profit_to_inventories', 'total_liabilities_to_equity',
    'profit_on_sales_to_total_assets', 'total_liabilities_to_cash',
    'operating_expenses_to_short_term_liabilities', 'operating_expenses_to_total_liabilities',
    'profit_to_total_assets', 'net_profit_to_total_assets_2',
    'total_liabilities_to_operating_cash', 'total_liabilities_to_total_assets_rotation',
    'profit_to_long_term_liabilities', 'total_assets_to_total_liabilities',
    'short_term_liabilities_to_long_term_liabilities', 'short_term_liabilities_to_equity_2',
    'long_term_liabilities_to_equity', 'sales_to_short_term_liabilities',
    'inventory_to_receivables', 'receivables_to_sales',
    'net_profit_to_total_assets_3', 'total_costs_to_total_sales',
    'long_term_liabilities_to_total_assets', 'sales_to_equity',
    'sales_to_total_assets_3', 'liabilities_to_working_capital',
    'operating_cash_to_liabilities', 'operating_cash_to_total_liabilities',
    'operating_cash_to_sales', 'sales_to_receivables',
    'operating_expenses_to_sales', 'total_costs_to_total_assets',
    'profit_on_operating_activities', 'depreciation_to_profit',
    'total_liabilities_to_profit', 'profit_on_operating_activities_to_financial_expenses',
    'working_capital_to_liabilities', 'equity_to_fixed_assets'
]
df_polish = df_polish.rename(columns={f'A{i+1}': column_names_64[i] for i in range(64)})

# force ratio columns numeric, else select_dtypes silently skips any that arrive as strings
for c in column_names_64:
    if c in df_polish.columns:
        df_polish[c] = pd.to_numeric(df_polish[c], errors='coerce')
df_polish['class'] = pd.to_numeric(df_polish['class'], errors='coerce').astype(int)

numeric_cols_polish = [c for c in df_polish.select_dtypes(include='number').columns
                       if c not in ['class', 'year', 'id']]
for col in numeric_cols_polish:
    df_polish[col] = df_polish[col].fillna(df_polish[col].median())
numeric_cols_polish = [c for c in numeric_cols_polish if df_polish[c].nunique() > 1]

print(f"Poland: {df_polish.shape}, {len(numeric_cols_polish)} usable features")
print(df_polish['class'].value_counts().to_dict())

## 2. Principal component regression

PCR runs PCA first, keeps the leading components, and fits the model on those instead of the original features.
Because PCA is unsupervised, it chooses directions by variance alone without reference to the target, which is
exactly the weakness PLSR addresses in the next section.

The baseline for comparison is a logistic regression on all features.

In [ ]:
def pcr_sweep(df, feats, target, name, ks=(5, 10, 20, 40)):
    X, y = df[feats], df[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    rows = []
    for k in ks:
        pipe = make_pipeline(qt(), PCA(n_components=k, random_state=RANDOM_STATE),
                             LogisticRegression(max_iter=3000)).fit(Xtr, ytr)
        var = pipe[1].explained_variance_ratio_.sum()
        rows.append({'components': k, 'variance retained': var,
                     'Test ROC-AUC': roc_auc_score(yte, pipe.predict_proba(Xte)[:, 1])})

    full = make_pipeline(qt(), LogisticRegression(max_iter=3000)).fit(Xtr, ytr)
    baseline = roc_auc_score(yte, full.predict_proba(Xte)[:, 1])

    res = pd.DataFrame(rows)
    print(f"\n=== {name}: principal component regression ({len(feats)} features) ===")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"   All {len(feats)} features: {baseline:.4f}")
    return res, baseline

pcr_pl, base_pl = pcr_sweep(df_polish, numeric_cols_polish, 'class', 'POLAND')
pcr_tw, base_tw = pcr_sweep(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')

PCR improves monotonically as components are added but never catches the full model in either dataset. Note
how much variance has to be retained to get close: even 40 components leave a gap.

This confirms what the Module B PCA predicted. When the variance is spread thinly across many directions, the
components you discard are not noise, they carry real signal.

## 3. Partial least squares regression

PLSR differs from PCA in one important way: it constructs components using the **target**, searching for
directions that explain variance in X *and* covary with y. For a supervised problem that should be strictly more
efficient, so PLSR ought to need fewer components than PCR to reach the same performance.

PLSRegression outputs a continuous score rather than a probability. For ranking purposes that is fine: ROC-AUC
only cares about ordering, so the raw prediction is used directly as the risk score.

In [ ]:
def plsr_sweep(df, feats, target, name, ks=(2, 5, 10, 20)):
    X, y = df[feats], df[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    scaler = qt()
    Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)

    rows = []
    for k in ks:
        pls = PLSRegression(n_components=k).fit(Xtr_s, ytr)
        rows.append({'components': k,
                     'Test ROC-AUC': roc_auc_score(yte, pls.predict(Xte_s).ravel())})
    res = pd.DataFrame(rows)
    print(f"\n=== {name}: partial least squares ===")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return res

pls_pl = plsr_sweep(df_polish, numeric_cols_polish, 'class', 'POLAND')
pls_tw = plsr_sweep(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')

PLSR behaves the way the theory says it should at low component counts: with only five components it already
beats PCR at five, because it is using the target to choose directions rather than variance alone.

But it then **peaks early and declines**. Beyond about five components the extra directions begin fitting
training noise, so more is actively worse. PCR, being unsupervised, has no such peak, it just improves slowly.
That contrast is the clearest practical difference between the two methods here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (pcr, pls, base, name) in zip(axes, [(pcr_pl, pls_pl, base_pl, 'Poland'),
                                             (pcr_tw, pls_tw, base_tw, 'Taiwan')]):
    ax.plot(pcr['components'], pcr['Test ROC-AUC'], marker='o', ms=6,
            color='steelblue', label='PCR')
    ax.plot(pls['components'], pls['Test ROC-AUC'], marker='s', ms=6,
            color='coral', label='PLSR')
    ax.axhline(base, ls='--', c='k', lw=1.3, label=f'all features ({base:.3f})')
    ax.set_xlabel('components retained'); ax.set_ylabel('Test ROC-AUC')
    ax.set_title(f'{name}: compression costs accuracy', fontsize=10)
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 4. Forward stepwise selection

Stepwise selection keeps the original features rather than building composites, which preserves
interpretability, a model using ten named financial ratios can be explained; one using ten principal components
cannot.

Forward selection starts empty and adds whichever feature most improves cross-validated score, repeating until
the target count is reached. It is greedy and therefore not guaranteed optimal, but it is tractable where
exhaustive search over 2^64 subsets is not. Poland is subsampled here purely for runtime, since each step refits
the model once per remaining candidate.

In [ ]:
def forward_selection(df, feats, target, name, k=10, n_sample=8000):
    d = df.sample(n_sample, random_state=RANDOM_STATE) if n_sample and n_sample < len(df) else df
    X, y = d[feats], d[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    sfs = SequentialFeatureSelector(LogisticRegression(max_iter=1000),
                                    n_features_to_select=k, direction='forward',
                                    scoring='roc_auc', cv=3, n_jobs=-1)
    make_pipeline(qt(), sfs).fit(Xtr, ytr)
    selected = [f for f, keep in zip(feats, sfs.get_support()) if keep]

    pipe = make_pipeline(qt(), LogisticRegression(max_iter=3000)).fit(Xtr[selected], ytr)
    auc = roc_auc_score(yte, pipe.predict_proba(Xte[selected])[:, 1])

    print(f"\n=== {name}: forward selection, {k} of {len(feats)} features ===")
    print(f"   Test ROC-AUC = {auc:.4f}")
    for i, f in enumerate(selected, 1):
        print(f"   {i:2d}. {f}")
    return selected, auc

sel_pl, auc_fwd_pl = forward_selection(df_polish, numeric_cols_polish, 'class', 'POLAND')
sel_tw, auc_fwd_tw = forward_selection(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN', n_sample=None)

### Selection is where these methods are actually useful

Forward selection reaches roughly 0.800 in Poland and 0.938 in Taiwan using **ten features instead of 64 and
94**. That costs about 0.015 of ROC-AUC against the full model, a small price for a 6- to 9-fold
reduction in model size.

The selected features are also readable, which the PCA components are not. In Taiwan the chosen set spans
per-share value, profitability persistence, and two debt-structure measures: distinct accounting families rather
than near-duplicates. This is the same redundancy-pruning behavior lasso showed in Week 2, arrived at by a
completely different mechanism, and it is a useful independent check that these datasets contain roughly
ten features' worth of distinct information.

In [ ]:
summary = pd.DataFrame({
    'Poland': {'All features': base_pl,
               'PCR (40 components)': pcr_pl['Test ROC-AUC'].iloc[-1],
               'PLSR (best)': pls_pl['Test ROC-AUC'].max(),
               'Forward selection (10)': auc_fwd_pl},
    'Taiwan': {'All features': base_tw,
               'PCR (40 components)': pcr_tw['Test ROC-AUC'].iloc[-1],
               'PLSR (best)': pls_tw['Test ROC-AUC'].max(),
               'Forward selection (10)': auc_fwd_tw},
})
print(summary.to_string(float_format=lambda x: f"{x:.4f}"))

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(summary)); w = 0.37
ax.bar(x - w/2, summary['Poland'], w, label='Poland', color='steelblue')
ax.bar(x + w/2, summary['Taiwan'], w, label='Taiwan', color='coral')
ax.set_xticks(x); ax.set_xticklabels(summary.index, fontsize=8.5)
ax.set_ylim(0.70, 1.0); ax.set_ylabel('Test ROC-AUC')
ax.set_title('No reduction method beats using every feature', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 5. Conclusions

1. **Compression costs accuracy in both datasets, as the Module B PCA predicted.** No PCR or PLSR configuration
   reached the all-features baseline. When PC1 captures only 15.3% and 13.4% of variance, the discarded
   components carry real signal rather than noise.

2. **PLSR is more efficient per component than PCR but peaks and then declines.** Using the target to construct
   components helps at low counts, and hurts beyond roughly five as the later components fit training noise. PCR,
   unsupervised, improves slowly and monotonically instead.

3. **Stepwise selection is the most useful of the three here, for interpretability rather than accuracy.** Ten
   named ratios reach within about 0.015 of the full model, and unlike principal components they can be explained
   to a credit committee.

4. **Three independent methods now agree these datasets hold roughly ten features' worth of distinct
   information.** Lasso (Week 2), VIF pruning (Week 1), and forward selection (this week) converge on that
   conclusion by different mechanisms.

**Carried forward:** dimensionality reduction will not be used to improve accuracy, but the forward-selected
ten-feature sets give a compact, interpretable model worth keeping as a communication tool, and a principled
starting feature list for the cross-country transfer experiment.